### Step 1: Document Ingestion & Indexing
- Parse both PDFs into clean text.
- Generate embeddings using an open-source model.
- Store in a vector database (e.g., FAISS, Chroma).
- Preserve metadata: document, section, page number.

In [3]:
# Imports
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from pinecone import ServerlessSpec


In [4]:
# Create a function to load and split the PDF documents
def load_n_split_pdf(path: str) -> list:
    loader = PyPDFLoader(path)
    docs = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = text_splitter.split_documents(docs)
    return chunks

In [5]:
# Load amd Split Apple & Tesla data
docs_apple = load_n_split_pdf("../documents/10-Q4-2024-As-Filed.pdf")
docs_tesla = load_n_split_pdf("../documents/tsla-20231231-gen.pdf")

In [6]:
len(docs_apple), len(docs_tesla)

(499, 533)

In [28]:
## Embedding

# Create Embeddings and Pinecone Index
embeddings = OllamaEmbeddings(model="mxbai-embed-large")
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

index_name = "finance-agent-index"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

In [ ]:
# import time
# # embeddings = OllamaEmbeddings(model="embeddinggemma") # 0.8507540225982666
# # embeddings = OllamaEmbeddings(model="llama3") # 8.896269083023071
# embeddings = OllamaEmbeddings(model="mxbai-embed-large") # 0.07343173027038574


# start = time.time()
# vector=embeddings.embed_query("Test sentence for benchmarking")
# print("Time per embedding:", time.time() - start)
# print(len(vector))

Time per embedding: 0.07343173027038574
1024


In [29]:
index = pc.Index(index_name)
index.describe_index_stats()

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [30]:
import langchain
langchain.debug = True

In [ ]:
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

# Add Apple documents to Pinecone index
print("Adding Apple documents to Pinecone index Started...")
vector_store.add_documents(
    documents=docs_apple, 
    batch_size=200)
print("Adding Apple documents to Pinecone index Completed...")

Adding Apple documents to Pinecone index Started...
Adding Apple documents to Pinecone index Completed...


In [32]:
index.describe_index_stats()

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 499}},
 'total_vector_count': 499,
 'vector_type': 'dense'}

In [33]:
# Add Tesla documents to Pinecone index
print("Adding Tesla documents to Pinecone index Started...")
vector_store.add_documents(
    documents=docs_tesla, 
    batch_size=200)
print("Adding Tesla documents to Pinecone index Completed...")

Adding Tesla documents to Pinecone index Started...
Adding Tesla documents to Pinecone index Completed...


In [34]:
index.describe_index_stats()

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1032}},
 'total_vector_count': 1032,
 'vector_type': 'dense'}

In [43]:
# import pprint
# pprint.pp(docs[0].metadata)

### Step 2: Build a Retrieval Pipeline \n
- Implement vector similarity search to retrieve top-5 relevant chunks.
- Add a re-ranker.
- Justify your choice in the design report

In [41]:
vector_store.similarity_search_with_score(
    query="what is the compensation received by any of the registrant’s executive officers during the relevant recovery period?",
    k=3
)

[(Document(id='ebd54d10-06c3-4995-a4ef-00c1f0c75c06', metadata={'author': 'anonymous', 'creationdate': '2024-10-31T19:17:25+00:00', 'creator': 'Workiva', 'moddate': '2024-10-31T14:07:33-07:00', 'page': 0.0, 'page_label': '1', 'producer': 'Wdesk Fidelity Content Translations Version 010.004.252', 'source': '../documents/10-Q4-2024-As-Filed.pdf', 'title': '10-K 2024, 09.28.2024-2024-10-31-12-16', 'total_pages': 121.0}, page_content='0.000% Notes due 2025 — The Nasdaq Stock Market LLC\n0.875% Notes due 2025 — The Nasdaq Stock Market LLC\n1.625% Notes due 2026 — The Nasdaq Stock Market LLC\n2.000% Notes due 2027 — The Nasdaq Stock Market LLC\n1.375% Notes due 2029 — The Nasdaq Stock Market LLC\n3.050% Notes due 2029 — The Nasdaq Stock Market LLC\n0.500% Notes due 2031 — The Nasdaq Stock Market LLC\n3.600% Notes due 2042 — The Nasdaq Stock Market LLC\nSecurities registered pursuant to Section 12(g) of the Act:  None\nIndicate by check mark if the Registrant is a well-known seasoned issuer, 

In [40]:
def retrieve_candidates(query: str, k: int = 20):
    docs = vector_store.similarity_search(query, k=k)
    return docs

In [41]:
import cohere

co = cohere.Client(os.getenv("COHERE_API_KEY"))

def rerank_documents(query: str, docs, top_n: int = 5):
    doc_texts = [doc.page_content for doc in docs]

    response = co.rerank(
        model="rerank-english-v3.0",
        query=query,
        documents=doc_texts,
        top_n=top_n
    )

    reranked_docs = [docs[result.index] for result in response.results]

    return reranked_docs

In [42]:
def retrieval_pipeline(query: str):
    # Step 1: Retrieve candidates
    candidates = retrieve_candidates(query, k=20)

    # Step 2: Rerank
    top_docs = rerank_documents(query, candidates, top_n=5)

    return top_docs

In [43]:
retrieval_pipeline("what is the compensation received by any of the registrant’s executive officers during the relevant recovery period?")

[Document(id='cb028317-5212-46c2-83db-79eac4519e08', metadata={'creationdate': '2024-01-29T11:11:14+00:00', 'creator': 'wkhtmltopdf 0.12.6', 'page': 1.0, 'page_label': '2', 'producer': 'Qt 5.15.2', 'source': '../documents/tsla-20231231-gen.pdf', 'title': '', 'total_pages': 130.0}, page_content='correction\tof\tan\terror\tto\tpreviously\tissued\tfinancial\tstatements.\t\no\nIndicate\tby\tcheck\tmark\twhether\tany\tof\tthose\terror\tcorrections\tare\trestatements\tthat\trequired\ta\trecovery\tanalysis\tof\tincentive-based\tcompensation\treceived\tby\tany\tof\tthe\nregistrant’s\texecutive\tofficers\tduring\tthe\trelevant\trecovery\tperiod\tpursuant\tto\t§240.10D-1(b).\t\no\nIndicate\tby\tcheck\tmark\twhether\tthe\tregistrant\tis\ta\tshell\tcompany\t(as\tdefined\tin\tRule\t12b-2\tof\tthe\tExchange\tAct).\tYes\t\no\n\tNo\t\nx\nThe\taggregate\tmarket\tvalue\tof\tvoting\tstock\theld\tby\tnon-affiliates\tof\tthe\tregistrant,\tas\tof\tJune\t30,\t2023,\tthe\tlast\tday\tof\tthe\tregistrant’s\tmos

### Step 3: LLM Integration
- Use a locally hosted or open-access LLM (e.g., Llama 3, Mistral, Phi-3).
- Do not use GPT-4, Claude, or any closed API.
- Design a custom prompt that:
    - Uses only retrieved context.
    - Cites sources as ["Apple 10-K", "Item 8", "p. 28"].
    - Responds with "Not specified in the document." if the answer is not in the provided files.
    - Refuses out-of-scope questions with: "This question cannot be answered based on the provided documents."

In [44]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3",
    temperature=0  # critical for factual grounding
)

In [46]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a financial document QA assistant.

You MUST follow these rules strictly:

1. Answer ONLY using the provided context.
2. Do NOT use external knowledge.
3. Every factual statement must be cited in this format:
   ["Document Name", "Section", "Page"].
4. If the answer is not explicitly found in the context, respond exactly:
   "Not specified in the document."
5. If the question is unrelated to the provided financial documents, respond exactly:
   "This question cannot be answered based on the provided documents."

----------------------
Context:
{context}
----------------------

Question:
{question}

Answer:
""")

In [47]:
def format_context(docs):
    formatted_chunks = []

    for doc in docs:
        citation = f'["{doc.metadata.get("document_name")}", "{doc.metadata.get("section")}", "{doc.metadata.get("page")}"]'
        
        chunk = f"""
Source: {citation}
Content:
{doc.page_content}
"""
        formatted_chunks.append(chunk)

    return "\n\n".join(formatted_chunks)

In [48]:
def rag_pipeline(query: str):
    # Step 1: Retrieve
    candidates = vector_store.similarity_search(query, k=20)

    # Step 2: Rerank
    top_docs = rerank_documents(query, candidates, top_n=5)

    # Step 3: Format context
    context = format_context(top_docs)

    # Step 4: Build prompt
    messages = prompt.format_messages(
        context=context,
        question=query
    )

    # Step 5: LLM call
    response = llm.invoke(messages)

    return response.content

### Step 4: Answer the Test Questions
- Run your system on the 13 questions below and output answers in the specified JSON format

In [49]:
query = "What was Apple's operating income in 2023?"

answer = rag_pipeline(query)
print(answer)

According to the provided context, Apple's operating income in 2023 was $114,301 ["Apple Inc.", "47", "49.0"].


In [51]:
query = "What was Apples total revenue for the fiscal year ended September 28, 2024?"
answer = rag_pipeline(query)
print(answer)

According to the provided financial documents, Apple's total net sales (revenue) for the fiscal year ended September 28, 2024 is $391,035. This information can be found in the "CONSOLIDATED STATEMENTS OF OPERATIONS" document, specifically on page 31.0.

["Apple Inc.", "None", "31.0"]


In [52]:
question_list = [
{"question_id": 1, "question": "What was Apples total revenue for the fiscal year ended September 28, 2024?"},
{"question_id": 2, "question": "How many shares of common stock were issued and outstanding as of October 18, 2024?"},
{"question_id": 3, "question": "What is the total amount of term debt (current + non-current) reported by Apple as of September 28, 2024?"},
{"question_id": 4, "question": "On what date was Apples 10-K report for 2024 signed and filed with the SEC?"},
{"question_id": 5, "question": "Does Apple have any unresolved staff comments from the SEC as of this filing? How do you know?"},
{"question_id": 6, "question": "What was Teslas total revenue for the year ended December 31, 2023?"},
{"question_id": 7, "question": "What percentage of Teslas total revenue in 2023 came from Automotive Sales (excluding Leasing)?"},
{"question_id": 8, "question": "What is the primary reason Tesla states for being highly dependent on Elon Musk?"},
{"question_id": 9, "question": "What types of vehicles does Tesla currently produce and deliver?"},
{"question_id": 10, "question": "What is the purpose of Teslas ’lease pass-through fund arrangements’?"},
{"question_id": 11, "question": "What is Teslas stock price forecast for 2025?"},
{"question_id": 12, "question": "Who is the CFO of Apple as of 2025?"},
{"question_id": 13, "question": "What color is Teslas headquarters painted?"}
]


In [ ]:
for item in question_list:
    print(f"Question ID- {item['question_id']}: {item['question']}")
    answer = rag_pipeline(item['question'])
    print(f"Answer: {answer}")
    print("-" * 50)

Question ID- 1: What was Apples total revenue for the fiscal year ended September 28, 2024?
Answer: According to the provided financial documents, Apple's total net sales (revenue) for the fiscal year ended September 28, 2024 is $391,035. This information can be found in the "CONSOLIDATED STATEMENTS OF OPERATIONS" document, specifically on page 31.0.

["Apple Inc.", "None", "31.0"]
--------------------------------------------------
Question ID- 2: How many shares of common stock were issued and outstanding as of October 18, 2024?
Answer: According to the context, ["None", "None", "1.0"] states: "15,115,823,000 shares of common stock were issued and outstanding as of October 18, 2024."
--------------------------------------------------
Question ID- 3: What is the total amount of term debt (current + non-current) reported by Apple as of September 28, 2024?
Answer: According to the provided context:

Source: ["None", "None", "33.0"]
Content:
Term debt  10,912  9,822 
Non-current liabiliti